# FSCT 8561 — Lab 1
## Socket Programming — Stateful Client-Server Chat Application

**Course:** FSCT 8561 — Security Applications  
**Lab Duration:** 3 hours

### Overview
This lab builds directly on Lab 0.

In Lab 0, your application used:

> **Connect → Send ONE message → Receive ONE reply → Close**

In Lab 1, you will transform that simple program into a **persistent, stateful client-server chat application**.

You will add:

- multiple messages,
- user input,
- a simple application-layer protocol,
- session state,
- validation,
- error handling,
- graceful disconnect,
- security analysis.


## Learning Objectives

By the end of this lab, you should be able to:

- design a simple application-layer protocol on top of TCP,
- maintain session state across multiple messages,
- keep a TCP connection open,
- use loops and conditional logic,
- validate incoming messages,
- handle malformed input and unexpected disconnects,
- terminate a session cleanly,
- analyze security implications of stateful network applications.


## Prerequisite

You must have completed **Lab 0**.

Start from the working `server.py` and `client.py` you created in Lab 0.


## Required Reading & Tutorials
- 	Mastering Python for Networking and Security – Chapter 3
    -    	https://learning.oreilly.com/library/view/mastering-python-for/9781839217166/
    -   	Source code on: https://github.com/PacktPublishing

-   Mastering-Python-for-Networking-and-Security-Second-Edition
- 	Socket programing tutorials
    - 	Official Python Documentation – socket Module
       -      https://docs.python.org/3/library/socket.html
    - 	Real Python – Socket Programming in Python
       -      https://realpython.com/python-sockets/
    - 	GeeksforGeeks – Socket Programming in Python
       -      https://www.geeksforgeeks.org/socket-programming-python/
    - 	DigitalOcean – How To Use Sockets in Python 3


# Part 1 — New Python Concepts

Lab 1 introduces a few new Python ideas.

### `input()`
Allows a user to type a value:


In [ ]:
message = input("Enter a message: ")
print("You typed:", message)


### `while True`
Repeats code until you explicitly stop it:


In [ ]:
count = 0

while True:
    print("Loop is running")
    count = count + 1

    if count == 3:
        break


### `if / elif / else`
Allows a program to make decisions:


In [ ]:
command = "HELLO"

if command == "HELLO":
    print("Start session")
elif command == "MSG":
    print("Process message")
else:
    print("Unknown command")


### `.split()`
Your protocol will use:

```text
COMMAND|DATA
```

You can separate it with:


In [ ]:
message = "HELLO|Maryam"

command, data = message.split("|", 1)

print("Command:", command)
print("Data:", data)


### `try / except`
Used to handle errors without immediately crashing:


In [ ]:
try:
    number = int("abc")
except ValueError:
    print("Invalid value")


# Part 2 — Start with Your Lab 0 Code

Before adding anything new:

1. Run your Lab 0 server.
2. Run your Lab 0 client.
3. Confirm one message is sent and one response is received.

Do not continue until Lab 0 works.


# Part 3 — Make the Connection Persistent

The key difference in Lab 1 is that the client and server should **not close after one message**.

You will use a loop.

Conceptually:

```text
Connect
   ↓
Receive message
   ↓
Send response
   ↓
Receive another message
   ↓
Send another response
   ↓
...
   ↓
EXIT
   ↓
Close
```


# Part 4 — Application-Level Protocol

All client messages must follow:

```text
COMMAND|DATA
```

Use these commands:

```text
HELLO|username
MSG|text
EXIT|
```

Examples:

```text
HELLO|Maryam
MSG|Hello everyone
MSG|How are you?
EXIT|
```

The server responds with:

```text
OK|
```

or

```text
ERROR|reason
```


# Part 5 — Build the Stateful Server

Create or update:

```text
server.py
```


In [ ]:
import socket

HOST = "127.0.0.1"
PORT = 12345

server_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)

server_socket.bind((HOST, PORT))
server_socket.listen(1)

print("Server is waiting for a connection...")

client_socket, client_address = server_socket.accept()

print("Connected by:", client_address)

username = None
connected = True

while connected:

    try:
        data = client_socket.recv(1024)

        if not data:
            print("Client disconnected unexpectedly")
            break

        message = data.decode()

        print("Received:", message)

        if "|" not in message:
            client_socket.send(
                "ERROR|Invalid command format".encode()
            )
            continue

        command, content = message.split("|", 1)

        if command == "HELLO":

            if content == "":
                client_socket.send(
                    "ERROR|Username required".encode()
                )
            else:
                username = content
                print("Username:", username)

                client_socket.send(
                    "OK|Hello ".encode() + username.encode()
                )

        elif command == "MSG":

            if username is None:
                client_socket.send(
                    "ERROR|HELLO required first".encode()
                )

            elif content == "":
                client_socket.send(
                    "ERROR|Message cannot be empty".encode()
                )

            elif len(content) > 200:
                client_socket.send(
                    "ERROR|Message too long".encode()
                )

            else:
                print(username + " says:", content)

                client_socket.send(
                    ("OK|Message received from " + username).encode()
                )

        elif command == "EXIT":

            client_socket.send(
                "OK|Goodbye".encode()
            )

            connected = False

        else:
            client_socket.send(
                "ERROR|Unknown command".encode()
            )

    except ConnectionResetError:
        print("Connection reset by client")
        break

client_socket.close()
server_socket.close()

print("Server closed")


## What the Server Now Does

Compared with Lab 0, the server now:

- stays connected,
- processes multiple messages,
- stores a username,
- knows whether `HELLO` occurred,
- validates commands,
- rejects invalid messages,
- handles an unexpected disconnect,
- exits cleanly after `EXIT|`.


# Part 6 — Build the Stateful Client

Update:

```text
client.py
```


In [ ]:
import socket

HOST = "127.0.0.1"
PORT = 12345

client_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)

client_socket.connect((HOST, PORT))

username = input("Enter your username: ")

hello_message = "HELLO|" + username

client_socket.send(
    hello_message.encode()
)

response = client_socket.recv(1024)

print("Server:", response.decode())

while True:

    message = input(
        "Enter message or type EXIT to leave: "
    )

    if message.upper() == "EXIT":

        client_socket.send(
            "EXIT|".encode()
        )

        response = client_socket.recv(1024)

        print("Server:", response.decode())

        break

    protocol_message = "MSG|" + message

    client_socket.send(
        protocol_message.encode()
    )

    response = client_socket.recv(1024)

    print("Server:", response.decode())

client_socket.close()

print("Disconnected")


# Part 7 — Test Normal Operation

Run the server first.

Then run the client.

Example session:

```text
Enter your username: Maryam
Server: OK|Hello Maryam

Enter message or type EXIT to leave: Hello
Server: OK|Message received from Maryam

Enter message or type EXIT to leave: How are you?
Server: OK|Message received from Maryam

Enter message or type EXIT to leave: EXIT
Server: OK|Goodbye
Disconnected
```


# Part 8 — Robustness Tests

Demonstrate at least **three** of the following:

1. Empty message
2. Message longer than 200 characters
3. Invalid command format
4. `MSG|...` sent before `HELLO`
5. Unexpected client disconnect
6. Unknown command

For some tests, you may temporarily modify the client code to send an invalid protocol message.

Example:

```python
client_socket.send("BADCOMMAND|test".encode())
```


# Part 9 — Why `recv(1024)` Is Not a Message Boundary

TCP is a **byte stream**.

This means:

```python
recv(1024)
```

does **not** guarantee that one complete application message arrives in one call.

A long message may be split across multiple `recv()` calls, or multiple small messages may arrive together.

For this lab, the simple protocol is acceptable for learning purposes, but a production system needs proper **message framing**.


# Part 10 — Security Reflection

Answer:

1. What happens if the server crashes while the client is connected?
2. How could the server be extended to handle multiple clients?
3. Why can `recv(1024)` split messages unexpectedly?
4. Why does TCP reliability not mean the application is secure?
5. What new attack surfaces are created by adding usernames and protocol commands?
6. How could a malicious client abuse the protocol?
7. How would you add authentication?
8. How would you add encryption?
9. What validation is performed in your implementation?
10. What additional mitigations would be needed for a real deployment?


# Part 11 — Raw TCP vs. Socket.IO

The textbook also shows Socket.IO examples.

Raw TCP sockets use operations such as:

```text
bind()
listen()
accept()
connect()
send()
recv()
```

Socket.IO hides many of these details and provides higher-level event-based communication.

Example concept:

```python
sio.emit("message", {"data": "hello"})
```

### task
Implement stateful chat server using Socket.IO and test it.

### Discussion
Which low-level operations from this lab are hidden by Socket.IO?

This comparison is for understanding abstraction. The required Lab 1 implementation should use **raw TCP sockets**.


# Challenge — Support Multiple Clients

Your current Lab 1 server supports only **one client at a time**. In a real chat application, multiple users should be able to connect to the same server.

For this challenge, extend your Lab 1 application so that the server can support **multiple clients concurrently**.

## Challenge Requirements

Modify your application so that:

1. At least **two clients** can connect to the server at the same time.
2. Each client provides a username using:

```text
HELLO|username
```

3. The server maintains the username associated with each connected client.
4. A client can send:

```text
MSG|Hello everyone!
```

5. The server forwards the message to the other connected client(s), including the sender's username:

```text
Maryam: Hello everyone!
```

6. `EXIT|` disconnects only that client while the server continues running for other users.
7. The server handles an unexpected client disconnect without crashing.

## Hint — Concurrency with Threads

Your current server becomes occupied while communicating with one client.

Research how Python's `threading` module can allow the server to handle each connected client independently.

You may begin by exploring:

```python
import threading
```

and:

```python
threading.Thread(...)
```

Do not simply copy a complete multi-client chat program. Your goal is to understand how concurrency changes the design of the Lab 1 server.

## Demonstration

Run the programs in three terminals:

```text
Terminal 1 → server.py

Terminal 2 → client.py → Alice

Terminal 3 → client.py → Bob
```

Demonstrate that Alice and Bob can exchange messages through the server.

## Challenge Reflection

Answer briefly:

1. Why could the original Lab 1 server handle only one client at a time?
2. What problem does threading solve?
3. What happens to the other clients when one client disconnects?
4. What additional security risks arise when multiple clients share the same server?
5. What could happen if two clients try to use the same username?
6. How would you prevent one user from impersonating another user?

## GitHub Organization

Keep the challenge separate from your original Lab 1 implementation:

```text
Lab1/
├── client.py
├── server.py
└── challenge/
    ├── client.py
    └── server.py
```

Your original Lab 1 code should remain unchanged so that your progression from a single-client server to a multi-client server is clear.


# Deliverables

Submit:

- `server.py`
- `client.py`
- a recording demonstrating:
  - connection,
  - multiple messages,
  - protocol behavior,
  - at least three robustness tests,
  - clean disconnection,
- a 300–400 word security analysis,
- reflection answers.

If you complete the **Challenge**, also include:

- `challenge/server.py`
- `challenge/client.py`
- screenshots or a short recording showing at least two clients connected at the same time,
- your answers to the Challenge Reflection questions.

Submit one PDF containing the security analysis, reflection answers, and links to technical artifacts.

### Filename

```text
Lab1-FirstName-LastName-StudentNumber.pdf
```
